# 📓 Notebook 07 — Tool Usage---**Phase 3 · Tools & Function Calling**> An LLM that can only generate text is a chatbot. An LLM that can USE TOOLS is an agent. This is the turning point.### What You'll Learn| Topic | Why It Matters ||-------|---------------|| Tool definition | JSON schema for tool interfaces || Execution loop | Call → Execute → Return → Final answer || Error handling | Graceful failures, retries, timeouts || Multiple tools | Routing between search, math, APIs || Safety | Preventing tool misuse |### 🏗️ Build: Agent with Search, Math, and API Tools

## 1. Setup

In [ ]:
import os, sys
import litellm

from setup_llm import DEFAULT_MODEL, is_proxy_mode

mode = "🔗 proxy" if is_proxy_mode() else "🔑 direct"
print(f"🤖 Model: {DEFAULT_MODEL} ({mode})")

---## 2. Tool Definitions — The Contract Between LLM and Code### The Anatomy of a Tool```json{  "type": "function",  "function": {    "name": "tool_name",           // Unique identifier    "description": "What it does", // LLM reads this to decide when to use it    "parameters": {                // JSON Schema for arguments      "type": "object",      "properties": { ... },      "required": [...]    }  }}```> **Interview Tip:** The `description` field is CRITICAL. It's the LLM's only guide for deciding when to call which tool. Vague descriptions = wrong tool selection.

In [ ]:
# Define a comprehensive tool kittools = [    {        "type": "function",        "function": {            "name": "search_web",            "description": "Search the web for current information. Use for questions about recent events, facts you're unsure about, or anything requiring up-to-date information.",            "parameters": {                "type": "object",                "properties": {                    "query": {"type": "string", "description": "The search query"},                    "num_results": {"type": "integer", "description": "Number of results to return (1-5)", "default": 3}                },                "required": ["query"]            }        }    },    {        "type": "function",        "function": {            "name": "calculate",            "description": "Perform mathematical calculations. Use for any arithmetic, algebra, or numerical computation. Supports +, -, *, /, **, sqrt, sin, cos, log.",            "parameters": {                "type": "object",                "properties": {                    "expression": {"type": "string", "description": "Mathematical expression to evaluate, e.g., '(15.5 * 3) + sqrt(144)'"}                },                "required": ["expression"]            }        }    },    {        "type": "function",        "function": {            "name": "get_current_time",            "description": "Get the current date and time in a specific timezone.",            "parameters": {                "type": "object",                "properties": {                    "timezone": {"type": "string", "description": "Timezone name, e.g., 'UTC', 'US/Eastern', 'Asia/Tokyo'", "default": "UTC"}                },                "required": []            }        }    },    {        "type": "function",        "function": {            "name": "convert_units",            "description": "Convert between measurement units. Supports temperature (C/F/K), distance (km/mi), weight (kg/lb), and currency (approximate).",            "parameters": {                "type": "object",                "properties": {                    "value": {"type": "number", "description": "The numeric value to convert"},                    "from_unit": {"type": "string", "description": "Source unit"},                    "to_unit": {"type": "string", "description": "Target unit"}                },                "required": ["value", "from_unit", "to_unit"]            }        }    }]print(f"🔧 Defined {len(tools)} tools:")for t in tools:    f = t["function"]    params = list(f["parameters"]["properties"].keys())    print(f"  • {f['name']}({', '.join(params)}) — {f['description'][:60]}...")

---## 3. Tool Implementations

In [ ]:
import mathfrom datetime import datetimedef search_web(query, num_results=3):    """Simulated web search. In production, use SerpAPI, Brave, or Tavily."""    # Simulation — replace with real API in production    simulated_results = {        "weather": [{"title": "Current Weather", "snippet": "Currently 22°C, partly cloudy, humidity 65%"}],        "python": [{"title": "Python 3.12", "snippet": "Python 3.12 released with 5% speed improvement."}],        "default": [{"title": "Search Result", "snippet": f"Results for: {query}"}],    }    for key in simulated_results:        if key in query.lower():            return json.dumps({"results": simulated_results[key][:num_results]})    return json.dumps({"results": simulated_results["default"][:num_results]})def calculate(expression):    """Safe mathematical calculation."""    try:        # Create safe math environment        safe_dict = {            "sqrt": math.sqrt, "sin": math.sin, "cos": math.cos,            "tan": math.tan, "log": math.log, "log10": math.log10,            "pi": math.pi, "e": math.e, "abs": abs, "round": round,            "pow": pow, "min": min, "max": max,        }        # Restrict to safe characters        cleaned = expression.replace("^", "**")        result = eval(cleaned, {"__builtins__": {}}, safe_dict)        return json.dumps({"expression": expression, "result": result})    except Exception as e:        return json.dumps({"error": str(e), "expression": expression})def get_current_time(timezone="UTC"):    """Get current date/time."""    now = datetime.utcnow()    return json.dumps({"datetime": now.isoformat(), "timezone": timezone})def convert_units(value, from_unit, to_unit):    """Unit conversion."""    conversions = {        ("celsius", "fahrenheit"): lambda v: v * 9/5 + 32,        ("fahrenheit", "celsius"): lambda v: (v - 32) * 5/9,        ("km", "miles"): lambda v: v * 0.621371,        ("miles", "km"): lambda v: v * 1.60934,        ("kg", "lb"): lambda v: v * 2.20462,        ("lb", "kg"): lambda v: v * 0.453592,        ("meters", "feet"): lambda v: v * 3.28084,        ("feet", "meters"): lambda v: v * 0.3048,    }    key = (from_unit.lower(), to_unit.lower())    if key in conversions:        result = conversions[key](value)        return json.dumps({"value": value, "from": from_unit, "to": to_unit, "result": round(result, 4)})    return json.dumps({"error": f"Unsupported conversion: {from_unit} → {to_unit}"})# Map names to functionsTOOL_REGISTRY = {    "search_web": search_web,    "calculate": calculate,    "get_current_time": get_current_time,    "convert_units": convert_units,}print("✅ All tool implementations ready")

---## 4. The Complete Tool Execution LoopThis is the most important pattern in agent development.

In [ ]:
class ToolAgent:    """An agent that uses tools to answer questions."""        def __init__(self, tools, tool_functions, model=None, system_prompt=None):        self.tools = tools        self.tool_functions = tool_functions        self.model = model or DEFAULT_MODEL        self.system_prompt = system_prompt or (            "You are a helpful assistant with access to tools. "            "Use tools when needed to provide accurate, current information. "            "If you can answer from your knowledge, do so directly."        )        self.call_count = 0        def run(self, user_query, max_tool_rounds=5, verbose=True):        """Execute the full tool loop."""        messages = [            {"role": "system", "content": self.system_prompt},            {"role": "user", "content": user_query}        ]                for round_num in range(max_tool_rounds):            # Ask LLM            response = litellm.completion(                model=self.model,                messages=messages,                tools=self.tools,                tool_choice="auto",                temperature=0,            )                        message = response.choices[0].message            self.call_count += 1                        # Check if LLM wants to use tools            if not message.tool_calls:                if verbose:                    print(f"  💬 Final answer (round {round_num + 1})")                return message.content                        # Execute each tool call            messages.append(message)            for tc in message.tool_calls:                func_name = tc.function.name                args = json.loads(tc.function.arguments)                                if verbose:                    print(f"  🔧 Round {round_num + 1}: {func_name}({json.dumps(args)[:60]})")                                # Execute with error handling                try:                    if func_name in self.tool_functions:                        result = self.tool_functions[func_name](**args)                    else:                        result = json.dumps({"error": f"Unknown tool: {func_name}"})                except Exception as e:                    result = json.dumps({"error": str(e)})                                messages.append({                    "role": "tool",                    "tool_call_id": tc.id,                    "content": result,                })                                if verbose:                    print(f"     → {result[:80]}...")                return "⚠️ Max tool rounds exceeded"# Create the agentagent = ToolAgent(tools, TOOL_REGISTRY)# Test with various queriesprint("🤖 Tool Agent Demo")print("=" * 60)test_queries = [    "What is 1547 * 23 + sqrt(625)?",    "What time is it in UTC?",    "Convert 100 kg to pounds",    "Search for the latest Python version",    "If I have a room that's 5.5 meters long, how many feet is that?",]for query in test_queries:    print(f"\n❓ {query}")    answer = agent.run(query)    print(f"📝 {answer}")    print("-" * 40)

---## 5. Error Handling & Retry Strategies

In [ ]:
class RobustToolAgent(ToolAgent):    """Agent with retry logic and error recovery."""        def run(self, user_query, max_tool_rounds=5, max_retries=2, verbose=True):        messages = [            {"role": "system", "content": self.system_prompt},            {"role": "user", "content": user_query}        ]                for round_num in range(max_tool_rounds):            response = litellm.completion(                model=self.model, messages=messages,                tools=self.tools, tool_choice="auto", temperature=0,            )            message = response.choices[0].message                        if not message.tool_calls:                return message.content                        messages.append(message)            for tc in message.tool_calls:                func_name = tc.function.name                args = json.loads(tc.function.arguments)                                # Retry logic                result = None                for attempt in range(max_retries + 1):                    try:                        if func_name not in self.tool_functions:                            result = json.dumps({"error": f"Tool '{func_name}' not found. Available: {list(self.tool_functions.keys())}"})                            break                                                result = self.tool_functions[func_name](**args)                        break                    except Exception as e:                        if attempt < max_retries:                            if verbose:                                print(f"  ⚠️ Retry {attempt + 1}: {e}")                            time.sleep(1)                        else:                            result = json.dumps({"error": f"Failed after {max_retries + 1} attempts: {str(e)}"})                                messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})                                if verbose:                    status = "✅" if "error" not in result else "⚠️"                    print(f"  {status} {func_name} → {result[:60]}")                return "Max rounds exceeded"robust_agent = RobustToolAgent(tools, TOOL_REGISTRY)print("🛡️ Robust Agent Test")print("=" * 40)answer = robust_agent.run("Calculate the hypotenuse of a right triangle with sides 3 and 4")print(f"📝 {answer}")

---## 📝 Interview Quiz — Tool Usage### Q1: How does the LLM "decide" which tool to use?<details><summary>💡 Answer</summary>The LLM reads the tool **descriptions** and **parameter schemas** provided in the request. Based on the user query, it:1. Determines if a tool is needed (or if it can answer from knowledge)2. Selects the most appropriate tool based on the description match3. Generates the required arguments as JSON**Key insight:** The LLM does NOT execute anything. It outputs structured intent (function name + args). Your code is the actual executor.**Tip for good tool selection:** Write clear, specific descriptions with examples of when to use the tool.</details>### Q2: What happens if the LLM generates invalid arguments?<details><summary>💡 Answer</summary>The execution fails. Defenses:1. **JSON Schema validation** — Validate args against the tool schema before execution2. **Try/catch** — Wrap tool execution in error handling3. **Error feedback** — Send the error back to the LLM as a tool result4. **Retry** — LLM can self-correct after seeing the error message5. **Pydantic models** — Validate complex argument structures</details>### Q3: What are the security risks of tool usage?<details><summary>💡 Answer</summary>1. **Prompt injection** — User tricks LLM into calling dangerous tools2. **Over-privileged tools** — Tool does more than it should (e.g., write access when read-only needed)3. **Data exfiltration** — Tool sends sensitive data to external APIs4. **Infinite loops** — LLM keeps calling tools endlessly5. **Resource exhaustion** — Expensive API calls, large file operations**Defenses:** Allowlists, rate limits, max rounds, confirmation for destructive actions, sandboxing.</details>### Q4: How would you add a new tool to an agent system?<details><summary>💡 Answer</summary>1. **Define the JSON schema** — name, description, parameters2. **Implement the function** — with error handling and return JSON3. **Register it** — add to the tool registry4. **Test in isolation** — verify the function works standalone5. **Test with LLM** — verify the LLM selects it correctly for relevant queries6. **Add safety** — input validation, rate limits, logging</details>### Q5: What is "tool choice" and when would you force a specific tool?<details><summary>💡 Answer</summary>`tool_choice` options:- `"auto"` — LLM decides (default, most flexible)- `"none"` — Never use tools- `"required"` — Must use at least one tool- `{"type": "function", "function": {"name": "X"}}` — Must use tool X**Force specific tool when:**- Building a pipeline where the step is known (e.g., always need DB lookup)- Testing a specific tool- The query format guarantees which tool is needed</details>

---## ✅ Notebook 07 Summary| Concept | Key Takeaway ||---------|-------------|| Tool schema | JSON definition with name, description, parameters || Execution loop | LLM → tool intent → your code executes → result → LLM → answer || tool_choice | auto/none/required/specific — controls tool selection || Error handling | Try/catch, retry with feedback, max rounds || Safety | Allowlists, rate limits, input validation || Description quality | Most impactful factor in tool selection accuracy |### ➡️ Next: [Notebook 08 — ReAct Agent](./08_react_agent.ipynb)